# Stage 1: build and upload the clean math SFT problem pool

Run all cells in a **CPU Colab runtime**. No GPU, model calls, or training.
Enable a write-capable **HF_TOKEN** in Colab Secrets. The final cell uploads
the cleaned dataset and audit reports to **Seungjun/clean-math-sft-pool**,
creating a **private** dataset by default. Change `hf_private` to publish publicly.

Inputs: the original 7,500 MATH training problems (EleutherAI mirror),
NuminaMath-1.5 (896,215 rows), and the original authors' s1K (1,000 rows).
All training sources are pinned to verified commits. The exact five evaluation
references are bundled from this repository's `english_eval_suite.json`.
The RL reference is the content-checked export in `configs/dapo.yaml`.

This is a substantial CPU/disk job: allow hours, several GB of free local disk,
and a persistent runtime. Runtime depends on symbolic parsing and the number
of near-duplicate candidates; there is no measured full-corpus runtime yet.
SQLite keeps the deduplication index on local disk. Google Drive stores the
completed artifact and upload receipt. An interrupted build reruns from the
start; cached Hugging Face downloads can be reused within the runtime.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'datasets==5.0.1', 'huggingface-hub==0.36.2', 'PyYAML==6.0.3',
    'math-verify[antlr4_13_2]==0.9.0', 'sympy==1.14.0',
    'latex2sympy2-extended==1.11.0', 'antlr4-python3-runtime==4.13.2',
    'mpmath==1.3.0', 'pytest==9.1.1', 'nbformat==5.11.1'])
from google.colab import userdata, drive
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError('Enable HF_TOKEN in Colab Secrets and allow notebook access')
drive.mount('/content/drive')

In [ ]:
# @title Install the bundled, tested project code (self-contained notebook)
import base64, io, sys, zipfile
from pathlib import Path
CODE_ROOT = Path("/content/lg-sft-pool-code")
CODE_ROOT.mkdir(parents=True, exist_ok=True)
BUNDLE = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as bundle:
    bundle.extractall(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
print("Project code ready:", CODE_ROOT)
(CODE_ROOT / "sft_pool_bundle_manifest.json").write_text('{"files_sha256": {"common/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "common/io.py": "2ef0ca7d6875e49b1b4818df13de59ff56e11ec83a675bdf1eb361143eab4e4b", "common/math_text.py": "fe6eaf3782935189e41a32637120244c9544da82dc5bd5e1a65265063b5227ae", "configs/dapo.yaml": "17f18c8a13911e18ceb806e08a9c32a818bd1fd94c87d184892744873daf5254", "configs/english_eval_suite.json": "ab74925185e3d87dfb9bdec56e05148cfe864c0781690e86c7dac039c5803788", "configs/sft_pool.yaml": "6df660a1d19d91cb6e52015579e9f1c9722a170456ce2f810bf6bd26d7c7c6af", "pipeline/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855", "pipeline/datasets.py": "cca3f37ad8a04b7add7c49eb39066f9c16899d63cff018bbc8468e0f744eacb1", "pipeline/sft_pool.py": "c8f5c9aa6ae6621c8a8be1a2f12b7324c2f0e308d5be56a9a9a01ce41bf2817a", "pipeline/sft_pool_hub.py": "d73dd2a4017fdf8c9faf11a7f797e9d4a47faf35c6e1f73b37709af10e2c7949", "tests/test_sft_pool.py": "f90231c38441e23d5120759ec48388a8a5cbf00101de3a4208ced588437149b8"}, "git_commit": "410e842feb27a97ef5c13ab72019f4b6baa0ece4", "includes_uncommitted_implementation": true}')

In [ ]:
import json, yaml
from pathlib import Path
from huggingface_hub import HfApi
CONFIG = yaml.safe_load((CODE_ROOT / 'configs/sft_pool.yaml').read_text())
CONFIG['hf_repo'] = 'Seungjun/clean-math-sft-pool'
CONFIG['hf_private'] = True
# Optional changes; all values are recorded in provenance.json.
CONFIG['dedup_jaccard'] = 0.8
CONFIG['coverage_threshold'] = 0.70
CONFIG['near_miss_lower'] = 0.50
CONFIG['reference_ngram'] = 8
CONFIG['exclude_numina_sources'] = ['amc_aime']
OUTPUT = Path('/content/lg-sft-pool-output')
DRIVE_ROOT = Path('/content/drive/MyDrive/lg-korean-aime/data/sft_pool')
# Authenticated identity only; never display or save the token.
print('Authenticated as:', HfApi(token=HF_TOKEN).whoami()['name'])
print('Upload destination:', CONFIG['hf_repo'])
print('Private:', CONFIG['hf_private'])

## Acceptance checks before the large build

Tests cover LaTeX normalization, embedded and short evaluation matches,
distinct/per-reference overlap counts, numeric variants, cross-source priority,
conflicting/equivalent answers, exact prefix matching, and count reconciliation.
The optional local MATH-500 fixture test is skipped here; the build also checks
verbatim and embedded matches against an actual pinned MATH-500 input.

In [ ]:
subprocess.check_call([sys.executable, '-m', 'pytest', '-q',
                       str(CODE_ROOT / 'tests/test_sft_pool.py')], cwd=CODE_ROOT)

## Build the pool and reports

Filters are conservative and auditable. s1 uses both type and source/domain
metadata, because some physics questions are labeled `math`. Unknown domains
are dropped. Only gold answers are extracted; reasoning traces are discarded.
Multiple required answers and free text are excluded.

Deduplication uses an exact, disk-backed Jaccard prefix join, an alternative to
MinHash/LSH without probabilistic candidate misses. Conflicting clusters are
quarantined in full. Normalization is lexical, not symbolic equivalence.

Evaluation and RL exclusion use separate inverted indexes over distinct
8-grams with coverage measured against one reference at a time. A supplemental
numeric-template check flags number-swapped copies even below 0.50 coverage;
the actual coverage is never inflated. These remain manual-review candidates.
Ordinary coverage near misses remain in the pool unless another stage removes them.

In [ ]:
import shutil
from pipeline.sft_pool_hub import run, sha256_file
from common.io import write_json
SUMMARY = run(CONFIG, CODE_ROOT, OUTPUT, HF_TOKEN)
print(json.dumps(SUMMARY, indent=2))
# Content-addressed Drive export, separate from the local SQLite work index.
run_id = sha256_file(OUTPUT / 'COMPLETE.json')[:16]
SAVED_OUTPUT = DRIVE_ROOT / run_id
shutil.copytree(OUTPUT, SAVED_OUTPUT, dirs_exist_ok=True)
print('Completed artifacts saved to:', SAVED_OUTPUT)

In [ ]:
import pandas as pd
from pipeline.sft_pool import iter_jsonl
from itertools import islice
display(pd.DataFrame(SUMMARY['counts']).T)
preview = list(islice(iter_jsonl(OUTPUT / 'pool.jsonl'), 5))
display(pd.DataFrame(preview)[['dataset_source', 'problem', 'gold_answer',
                               'eval_max_coverage', 'rl_max_coverage']] if preview else 'Empty pool')
for kind in ['eval', 'rl']:
    print(kind, 'near-miss examples:')
    display(pd.DataFrame(list(islice(iter_jsonl(OUTPUT / f'{kind}_near_misses.jsonl'), 10))))

## Upload the completed artifacts to Hugging Face

Running this cell publishes `pool.jsonl`, the summary, all audit reports,
and provenance to **Seungjun/clean-math-sft-pool**. It verifies local checksums,
publishes one Hub commit, then verifies every uploaded file at that revision.
Existing visibility must match `hf_private`. Re-running updates this dataset;
source datasets and evaluation/RL references are never written.

In [ ]:
from pipeline.sft_pool_hub import upload
RECEIPT = upload(OUTPUT, CONFIG['hf_repo'], HF_TOKEN, private=CONFIG['hf_private'])
write_json(SAVED_OUTPUT / 'upload_receipt.json', RECEIPT)
print('Verified upload:', RECEIPT['url'])
print('Pinned revision:', RECEIPT['revision'])
print('Drive receipt:', SAVED_OUTPUT / 'upload_receipt.json')